# WS4 — GRPO Training (Final)
FarmSimulation hackathon notebook.

In [ ]:
!pip install -q unsloth vllm \
trl==0.22.2 \
transformers==4.56.2 \
huggingface_hub==0.34.0 \
openenv-core wandb

In [ ]:
from huggingface_hub import login
import os
login(os.environ.get("HF_TOKEN", "hf_token_here"))
import wandb
wandb.login()

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length=1104,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.7,
    enforce_eager=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Cell 5: FarmEnvClient
import requests
from typing import Dict, Any

class FarmEnvClient:
    def __init__(self, base_url: str) -> None:
        self.base_url = base_url.rstrip("/")
        self._s = requests.Session()

    def health(self) -> bool:
        try:
            return self._s.get(f"{self.base_url}/health", timeout=5).status_code == 200
        except Exception:
            return False

    @staticmethod
    def _unwrap(raw: Dict[str, Any]) -> Dict[str, Any]:
        if "observation" in raw:
            flat = dict(raw["observation"])
            flat["reward"] = raw.get("reward", 0.0)
            flat["done"] = raw.get("done", False)
            flat["metadata"] = raw.get("metadata", {})
            return flat
        return raw

    def reset(self, task_id: int = 1, noise_seed: int = 42) -> Dict[str, Any]:
        r = self._s.post(f"{self.base_url}/reset", json={"task_id": task_id, "noise_seed": noise_seed}, timeout=30)
        r.raise_for_status()
        return self._unwrap(r.json())

    def step(self, action: Dict[str, Any]) -> Dict[str, Any]:
        r = self._s.post(f"{self.base_url}/step", json={"action": action}, timeout=30)
        r.raise_for_status()
        return self._unwrap(r.json())

In [ ]:
# Cell 6: Action Parser
import json
import re

_VALID_ACTION_TYPES = {
    "wait", "buy_seeds", "plant", "irrigate", "harvest", "sell",
    "pump_water", "apply_fertilizer", "spray_pesticide", "pull_weeds",
    "buy_plot", "clear", "write_journal", "end_day",
}
_FALLBACK_ACTION = {"action_type": "wait"}
_JSON_RE = re.compile(r"\{[^{}]*\}", re.DOTALL)

def parse_action(text: str) -> Dict[str, Any]:
    if not text or not text.strip():
        return dict(_FALLBACK_ACTION)
    try:
        obj = json.loads(text.strip())
        if isinstance(obj, dict):
            return obj.get("action", obj) if "action" in obj else obj
    except json.JSONDecodeError:
        pass
    for m in _JSON_RE.finditer(text):
        try:
            obj = json.loads(m.group(0))
            if isinstance(obj, dict):
                return obj.get("action", obj) if "action" in obj else obj
        except json.JSONDecodeError:
            continue
    return dict(_FALLBACK_ACTION)

In [ ]:
# Cell 7: Reward Functions
import torch
import textwrap
import sys
sys.path.append('..')
from server.tasks import grade_episode_detailed, EpisodeRecord

SPACE_URL = "https://athric-farmsim.hf.space"

SYSTEM_PROMPT = textwrap.dedent("""\
    You are an autonomous farm manager. Each turn you observe the farm state and pick ONE action.
    Goal: maximize net worth via survival, growth, and well-timed market sales.
    Priorities: keep crops alive (irrigate when moisture is low), plant when you have seeds, harvest when mature, sell when prices are above the 7-day average.
    Reply with EXACTLY one JSON object: {"action_type": "...", "plot_id": 0, "seed_type": "...", "quantity": 1}.
    Omit fields that don't apply. Valid action_type values: wait, buy_seeds, plant, irrigate, harvest, sell, pump_water, apply_fertilizer, spray_pesticide, pull_weeds, buy_plot, clear, write_journal, end_day.""")

_ROLLOUT_CACHE = {}

def get_rollout_grade(completion_text, task_id, noise_seed):
    cache_key = (completion_text, task_id, noise_seed)
    if cache_key in _ROLLOUT_CACHE:
        return _ROLLOUT_CACHE[cache_key]
        
    first_action = parse_action(completion_text)
    env = FarmEnvClient(SPACE_URL)
    
    try:
        obs = env.reset(task_id=task_id, noise_seed=noise_seed)
        obs = env.step(first_action)
        
        steps = 1
        history = [f"Action: {first_action.get('action_type')}"]
        
        while not obs.get("done", False) and steps < 30:
            obs_text = obs.get("text_summary", obs.get("narrative_text", ""))
            valid = obs.get("valid_actions", [])
            user_msg = f"STATE:\n{obs_text}\n\nVALID ACTIONS THIS STEP: {', '.join(valid)}\n\nRECENT HISTORY:\n{chr(10).join(history[-3:])}\n\nReply with one JSON action."
            
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ]
            chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(chat, return_tensors="pt").to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=60, do_sample=False)
                
            action_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
            action = parse_action(action_text)
            obs = env.step(action)
            history.append(f"Action: {action.get('action_type')}")
            steps += 1

        record_dict = obs.get("metadata", {}).get("episode_record")
        if record_dict:
            record = EpisodeRecord(**record_dict)
            grade = grade_episode_detailed(record)
        else:
            grade = {"score": 0.01, "dimensions": {}, "gated": "missing"}
    except Exception as e:
        grade = {"score": 0.01, "dimensions": {}, "gated": str(e)}
        
    _ROLLOUT_CACHE[cache_key] = grade
    return grade

def task_completion_reward(prompts, completions, **kwargs):
    rewards = []
    task_ids = kwargs.get("task_id", [1]*len(completions))
    seeds = kwargs.get("noise_seed", [42]*len(completions))
    for completion, t_id, seed in zip(completions, task_ids, seeds):
        text = completion[0]["content"]
        grade = get_rollout_grade(text, t_id, seed)
        rewards.append(grade["score"] * 2.0)
    return rewards

def format_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        action = parse_action(text)
        if action.get("action_type") != "wait" or "wait" in text.lower():
            rewards.append(0.3)
        else:
            rewards.append(0.0)
    return rewards

def stewardship_reward(prompts, completions, **kwargs):
    rewards = []
    task_ids = kwargs.get("task_id", [1]*len(completions))
    seeds = kwargs.get("noise_seed", [42]*len(completions))
    for completion, t_id, seed in zip(completions, task_ids, seeds):
        text = completion[0]["content"]
        grade = get_rollout_grade(text, t_id, seed)
        rewards.append(grade.get("dimensions", {}).get("stewardship", 0.0) * 0.5)
    return rewards

def anti_exploit_reward(prompts, completions, **kwargs):
    rewards = []
    task_ids = kwargs.get("task_id", [1]*len(completions))
    seeds = kwargs.get("noise_seed", [42]*len(completions))
    for completion, t_id, seed in zip(completions, task_ids, seeds):
        text = completion[0]["content"]
        grade = get_rollout_grade(text, t_id, seed)
        if grade.get("gated"):
            rewards.append(-1.0)
        else:
            rewards.append(0.0)
    return rewards

In [ ]:
# Cell 8: Build the prompt dataset
from datasets import Dataset

print("Sampling 200 initial states from the environment...")
dataset_rows = []
env = FarmEnvClient(SPACE_URL)

try:
    for seed in range(200):
        obs = env.reset(task_id=1, noise_seed=seed)
        text_summary = obs.get("text_summary", obs.get("narrative_text", "(no summary)"))
        valid_actions = ", ".join(obs.get("valid_actions", []))
        
        user_msg = f"STATE:\n{text_summary}\n\nVALID ACTIONS THIS STEP: {valid_actions}\n\nRECENT HISTORY:\n(none)\n\nReply with one JSON action."
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ]
        
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        dataset_rows.append({
            "prompt": formatted_prompt,
            "task_id": 1,
            "noise_seed": seed
        })
except Exception as e:
    print(f"Make sure server is running at {SPACE_URL} -> Error: {e}")

if dataset_rows:
    prompt_dataset = Dataset.from_list(dataset_rows)
    print(f"Created prompt dataset with {len(prompt_dataset)} rows.")

In [ ]:
# Cell 9: GRPOConfig
from trl import GRPOConfig

training_args = GRPOConfig(
    temperature=1.0,
    max_prompt_length=1024,
    max_completion_length=80,
    num_generations=4,
    learning_rate=5e-6,
    optim="adamw_8bit",
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=50,
    beta=0.0,
    epsilon=0.2,
    epsilon_high=0.28,
    loss_type="dapo",
    mask_truncated_completions=True,
    logging_steps=1,
    save_steps=10,
    output_dir="grpo_farm_qwen_0_5b_gen2",
    report_to="wandb",
)

In [ ]:
# Cell 10: GRPOTrainer
from trl import GRPOTrainer

if 'prompt_dataset' in locals():
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[
            task_completion_reward,
            format_reward,
            anti_exploit_reward,
            stewardship_reward,
        ],
        args=training_args,
        train_dataset=prompt_dataset,
    )
    # trainer.train()
    print("Trainer ready. Uncomment trainer.train() to start!")
else:
    print("prompt_dataset not created. Run Cell 8 with the server running.")

In [ ]:
# Cell 11: Plotting
!pip install matplotlib pandas

import matplotlib.pyplot as plt
import pandas as pd
import os

os.makedirs("assets", exist_ok=True)
if 'trainer' in locals() and trainer.state.log_history:
    df = pd.DataFrame(trainer.state.log_history)
    reward_cols = [c for c in df.columns if 'reward' in c and 'margin' not in c]
    if reward_cols:
        reward_df = df.dropna(subset=reward_cols, how='all')
        plt.figure(figsize=(10, 6))
        for col in reward_cols:
            plt.plot(reward_df['step'], reward_df[col], label=col, marker='o', markersize=3)
        plt.title("GRPO Reward Evolution (Qwen 0.5B + FarmSim)")
        plt.xlabel("Update Step")
        plt.ylabel("Reward Value")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig("assets/grpo_rewards.png", dpi=300, bbox_inches='tight')
        print("Saved reward plot to assets/grpo_rewards.png")

In [ ]:
# Cell 12: Save and Push to Hub
HF_USERNAME = "Athric"
REPO_ID = f"{HF_USERNAME}/qwen-0.5b-farmsim-grpo-gen2"

print("Saving locally to grpo_farm_qwen_0_5b_gen2_final...")
model.save_pretrained("grpo_farm_qwen_0_5b_gen2_final")
tokenizer.save_pretrained("grpo_farm_qwen_0_5b_gen2_final")

try:
    model.push_to_hub(REPO_ID, use_auth_token=True)
    tokenizer.push_to_hub(REPO_ID, use_auth_token=True)
    print(f"Successfully pushed Gen 2 adapter to Hub! URL: https://huggingface.co/{REPO_ID}")
except Exception as e:
    print("Push failed:", e)


In [ ]:
# Cell 13: Metadata
import torch
import unsloth
import trl

print("="*50)
print("🌾 FARMSIMULATION GRPO PIPELINE METADATA")
print("="*50)
print(f"PyTorch Version:  {torch.__version__}")
print(f"Unsloth Version:  {unsloth.__version__}")
print(f"TRL Version:      {trl.__version__}")
print(f"GPU Used:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Max Steps:        {training_args.max_steps}")
print(f"Loss Type:        {training_args.loss_type}")
print(f"Hardware Opt:     vLLM Fast Inference + 8-bit AdamW")
print("="*50)